# 7. Reading coefficients in genomics

**Learning goals**: what a penalised coefficient does and does not mean; signs and standardisation;
why "selected" is not "significant"; and how to sanity-check a model against known biology.

* A coefficient is the effect of a gene **holding the other genes fixed**. With thousands of
  correlated genes, that conditional effect is rarely what a biologist means by "this gene matters".
* Penalised coefficients are **biased towards zero**, more for the lasso's small coefficients, less
  for the adaptive lasso's large ones.
* Here the response is AUC: a **negative** coefficient means higher expression, lower AUC, i.e. a
  more **sensitive** cell line.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (8, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(7)

In [2]:
# The project's data: one drug, its screened cell lines, 2,000 most variable genes
from drugresponse.experiment import Store
from drugresponse.models import Design
S = Store()
drug_name = "Nutlin-3a (-)"
d = S.drugs.index[S.drugs.drug_name == drug_name][0]
ok = S.R[d].notna().values
des = Design(2000).fit(S.Ev[ok], S.tissue[ok], S.sanger[ok])
X = des.transform(S.Ev[ok], S.tissue[ok], S.sanger[ok])[:, ~des.is_tissue()]   # genes only
y = S.R[d].values[ok]
genes = S.genes[des.genes_]
print(f"{drug_name}: {X.shape[0]} cell lines x {X.shape[1]} genes")

Nutlin-3a (-): 943 cell lines x 2000 genes


In [3]:
from scipy import stats
from sklearn.linear_model import ElasticNetCV
m = ElasticNetCV(l1_ratio=[0.2, 0.8], cv=5, alphas=30, max_iter=5000).fit(X, y)
coef = pd.Series(m.coef_, index=genes)
sel = coef[coef != 0].sort_values()
marg = pd.Series([stats.spearmanr(X[:, j], y).statistic for j in range(X.shape[1])], index=genes)
tbl = pd.DataFrame({"elastic-net coef": sel, "marginal Spearman with AUC": marg[sel.index]})
print(f"elastic net (mixing {m.l1_ratio_}): {len(sel)} genes selected")
tbl.head(12).round(3)

elastic net (mixing 0.8): 110 genes selected


,elastic-net coef,marginal Spearman with AUC
CDKN1A,-0.017,-0.345
PHLDA3,-0.014,-0.271
TRIM22,-0.009,-0.276
GLDC,-0.007,-0.135
ACTA2,-0.006,-0.307
ITGAL,-0.006,-0.103
PURPL,-0.005,-0.349
CACNA2D4,-0.004,-0.092
ENPP2,-0.003,-0.106
ZYX,-0.003,-0.135


For Nutlin-3a the most negative coefficients should belong to TP53 target genes, the
transcriptional readout of wild-type TP53. Lines with an active TP53 pathway respond to MDM2
inhibition. The marginal correlation has the same sign, so the multivariable model agrees with
the simple univariate picture. When the two disagree in sign, it is usually because a correlated
neighbour has taken the weight.

## Selected is not significant
A gene being selected does not come with a p-value. Valid post-selection inference needs sample
splitting or selective-inference methods (Lee et al., 2016). For a predictive model the honest
evidence is out-of-sample accuracy, plus stability of the selected set across folds, plus agreement
with known biology. The project reports all three (notebooks/04).

**Key takeaways**
* Interpret signs on the response's scale (here: negative = sensitising).
* Compare multivariable coefficients with marginal associations; disagreements point to correlation.
* Validate with known biomarkers before trusting new ones.